# Relatório de Auditoria e Qualidade dos Dados

**Projeto:** ANEEL - Energia em Risco: Análise de Dados de Continuidade Elétrica e Previsão de Risco Regulatório

**Metodologia:** CRISP-DM

**Período de Análise:** 2021 – 2025

---

## Visão Geral

Este notebook faz parte da **Fase 2 do CRISP-DM - Compreensão dos Dados (Data Understanding)** e valida a conformidade dos dados da camada **interim (`data/interim/`)** com o contrato de dados do projeto ANEEL (2021–2025). O objetivo é verificar schema, tipos, filtros de escopo, nulos, conversões, granularidade, duplicidades e relacionamentos.

---

## Sumário

- **01.** Configurações do ambiente
- **02.** Arquivos e camadas avaliadas
- **03.** Regras derivadas do contrato de dados
- **04.** Validação de schema e tipos
- **05.** Validação de filtros de escopo
- **06.** Validação de nulos e conversões
- **07.** Validação de granularidade e duplicidades
- **08.** Validação de chaves e relacionamentos
- **09.** Validações específicas por dataset
- **10**. Problemas encontrados e ações recomendadas

---


## 1. Configurações do ambiente

In [1]:
# Importações e configurações do ambiente
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd

# Configurações de diretórios e constantes
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CONT_PATH, INT_PATH, LIM_PATH, ATR_PATH, REG_PATH
from src.data.constants import ANO_INICIO, ANO_FIM

METADATA_DIR = PROJECT_ROOT / "docs" / "metadata"
os.makedirs(METADATA_DIR, exist_ok=True)


# Conexão DuckDB
con = duckdb.connect()

print(f"Período analisado: {ANO_INICIO}-{ANO_FIM}")

Período analisado: 2021-2025


In [2]:
# Configurações de exibição e estilo
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda value: f"{value:.2f}")

In [3]:
# Configurando variáveis de ambiente para armazenamento de figuras e metadados

TABLE_CONTRACT_RULES = METADATA_DIR / "n01_table_contract_rules.csv"
TABLE_SCHEMA_REPORT = METADATA_DIR / "n01_table_schema_report.csv"
TABLE_SCOPE_REPORT = METADATA_DIR / "n01_table_scope_report.csv"
TABLE_NULL_REPORT = METADATA_DIR / "n01_table_null_report.csv"
TABLE_DUPLICATES_REPORT = METADATA_DIR / "n01_table_duplicates_report.csv"
TABLE_RELATIONSHIP_REPORT = METADATA_DIR / "n01_table_relationship_report.csv"
TABLE_COMPLEMENTAR_VALIDATIONS = METADATA_DIR / "n01_table_complementar_validations.csv"
TABLE_ISSUES_REPORT = METADATA_DIR / "n01_table_issues_report.csv"

---

## 2. Verificação de arquivos

Nesta seção, foi realizado o inventário dos arquivos da camada `interim` (Camada Silver) avaliando o tamanho dos arquivos, número de linhas e colunas.

In [4]:
# Inventário dos arquivos da camada interim
DATASETS = {
    "continuidade": Path(CONT_PATH),
    "interrupcoes": Path(INT_PATH),
    "limites": Path(LIM_PATH),
    "atributos": Path(ATR_PATH),
    "regiao": Path(REG_PATH),
}

inventory = []
for name, path in DATASETS.items():
    exists = path.is_file()
    rows = (
        int(
            con.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(path)]).fetchone()[
                0
            ]
        )
        if exists
        else 0
    )
    description = {
        row[0]: row[1]
        for row in con.execute(
            "DESCRIBE SELECT * FROM read_parquet(?)", [str(path)]
        ).fetchall()
    }
    inventory.append(
        {
            "dataset": name,
            "camada": "interim",
            "arquivo": path.name,
            "existe": exists,
            "tamanho_mb": path.stat().st_size / (1024 * 1024) if exists else 0,
            "linhas": rows,
            "colunas": len(description),
        }
    )

df_inventory = pd.DataFrame(inventory)
display(df_inventory)

,dataset,camada,arquivo,existe,tamanho_mb,linhas,colunas
0,continuidade,interim,continuidade_2021_2025.parquet,True,2.48,375082,8
1,interrupcoes,interim,interrupcoes_2021_2025.parquet,True,694.11,44410820,15
2,limites,interim,limites_2021_2025.parquet,True,0.10,31844,6
3,atributos,interim,atributos.parquet,True,0.10,7569,9
4,regiao,interim,regiao.parquet,True,0.15,15162,5


---

## 3. Regras derivadas do contrato de dados

Nesta seção serão avaliadas as regras derivadas do contrato de dados. As validações reproduzem as regras do quality gate da camada `interim`: existência dos arquivos, verificação de dados nulos, schema mínimo, intervalo de anos, nulos críticos, valores inválidos e duplicidades de continuidade.

In [ ]:
# Regras centrais e granularidades esperadas

full_table_path = TABLE_CONTRACT_RULES
relative_table_path = os.path.relpath(full_table_path)

contract_rules = pd.DataFrame(
    [
        [
            "continuidade",
            "schema/domínio",
            "DEC/FEC, ano no período, mês 1-12, valor >= 0",
            "IdeConjunto + AnoIndice + NumPeriodoIndice + SigIndicador",
        ],
        [
            "interrupcoes",
            "schema/domínio",
            "datas válidas, duração >= 0, consumidores >= 0",
            "uma linha por evento",
        ],
        [
            "limites",
            "schema/domínio",
            "DEC/FEC, ano no período, limite numérico",
            "NumCNPJ + IdeConjunto + SigIndicador + AnoIndice",
        ],
        [
            "atributos",
            "schema/domínio",
            "IdeConjunto não nulo",
            "NumCNPJ + IdeConjunto",
        ],
        [
            "regiao",
            "schema/domínio",
            "IdeConjunto não nulo e região compatível com UF",
            "uma linha por IdeConjunto",
        ],
    ],
    columns=["dataset", "categoria", "regra", "granularidade"],
)

display(
    contract_rules.style.set_properties(
        subset=["granularidade"],
        **{"white-space": "normal", "min-width": "320px"},
    )
)

contract_rules.to_csv(
    full_table_path,
    index=False,
)

print(f"\nTabela salva com sucesso em: {relative_table_path}")

,dataset,categoria,regra,granularidade
0,continuidade,schema/domínio,"DEC/FEC, ano no período, mês 1-12, valor >= 0",IdeConjunto + AnoIndice + NumPeriodoIndice + SigIndicador
1,interrupcoes,schema/domínio,"datas válidas, duração >= 0, consumidores >= 0",uma linha por evento
2,limites,schema/domínio,"DEC/FEC, ano no período, limite numérico",NumCNPJ + IdeConjunto + SigIndicador + AnoIndice
3,atributos,schema/domínio,IdeConjunto não nulo,NumCNPJ + IdeConjunto
4,regiao,schema/domínio,IdeConjunto não nulo e região compatível com UF,uma linha por IdeConjunto



Tabela salva com sucesso em: ..\docs\metadata\n01_table_contract_rules.csv


---

## 4. Validação de schema e tipos

Nesta seção, são observadas as colunas e tipos mínimos esperados para cada dataset, bem como a granularidade de cada um. 

As regras do contrato são definidas para garantir a consistência e integridade dos dados, permitindo identificar quaisquer desvios ou inconsistências que possam surgir durante o processamento e análise dos dados.

In [ ]:
# Colunas e tipos mínimos previstos para cada arquivo interim

full_table_path = TABLE_SCHEMA_REPORT
relative_table_path = os.path.relpath(full_table_path)

DEFAULT_SCHEMA = {
    "continuidade": {
        "AnoIndice": {"INTEGER", "BIGINT"},
        "NumPeriodoIndice": {"INTEGER", "BIGINT"},
        "SigAgente": {"VARCHAR"},
        "NumCNPJ": {"VARCHAR"},
        "IdeConjunto": {"BIGINT", "INTEGER"},
        "DscConjunto": {"VARCHAR"},
        "SigIndicador": {"VARCHAR"},
        "VlrIndiceEnviado": {"DOUBLE", "FLOAT"},
    },
    "interrupcoes": {
        "AnoIndice": {"INTEGER", "BIGINT"},
        "NumPeriodoIndice": {"INTEGER", "BIGINT"},
        "DatInicioInterrupcao": {"TIMESTAMP"},
        "DatFimInterrupcao": {"TIMESTAMP"},
        "DuracaoHoras": {"DOUBLE", "FLOAT"},
        "SigAgente": {"VARCHAR"},
        "NumCNPJ": {"VARCHAR"},
        "IdeConjunto": {"BIGINT", "INTEGER"},
        "NumOrdemInterrupcao": {"VARCHAR"},
        "NumConsumidorConjunto": {"BIGINT", "INTEGER"},
    },
    "limites": {
        "SigAgente": {"VARCHAR"},
        "NumCNPJ": {"VARCHAR"},
        "IdeConjunto": {"BIGINT", "INTEGER"},
        "SigIndicador": {"VARCHAR"},
        "AnoIndice": {"INTEGER", "BIGINT"},
        "VlrLimite": {"DOUBLE", "FLOAT"},
    },
    "atributos": {
        "DatGeracaoConjuntoDados": {"DATE"},
        "SigAgente": {"VARCHAR"},
        "NumCNPJ": {"VARCHAR"},
        "IdeConjunto": {"BIGINT", "INTEGER"},
        "DscConjunto": {"VARCHAR"},
    },
    "regiao": {
        "IdeConjunto": {"BIGINT", "INTEGER"},
        "CodMunicipio": {"BIGINT", "INTEGER"},
        "Municipio": {"VARCHAR"},
        "UF": {"VARCHAR"},
        "Regiao": {"VARCHAR"},
    },
}

schema_results = []
for name, path in DATASETS.items():
    description = {
        row[0]: row[1]
        for row in con.execute(
            "DESCRIBE SELECT * FROM read_parquet(?)", [str(path)]
        ).fetchall()
    }
    missing_cols = sorted(set(DEFAULT_SCHEMA[name]) - set(description))
    incompatible_types = sorted(
        f"{coluna}: esperado {sorted(tipos)}, encontrado {description[coluna]}"
        for coluna, tipos in DEFAULT_SCHEMA[name].items()
        if coluna in description and description[coluna] not in tipos
    )
    schema_results.append(
        {
            "dataset": name,
            "colunas_identificadas": len(description),
            "colunas_faltantes": ", ".join(missing_cols) or "0",
            "tipos_incompativeis": "; ".join(incompatible_types) or "0",
        }
    )

df_schema = pd.DataFrame(schema_results)
display(df_schema)

df_schema.to_csv(
    full_table_path,
    index=False,
)

print(f"\nTabela salva com sucesso em: {relative_table_path}")

,dataset,colunas_identificadas,colunas_faltantes,tipos_incompativeis
0,continuidade,8,0,0
1,interrupcoes,15,0,0
2,limites,6,0,0
3,atributos,9,0,0
4,regiao,5,0,0



Tabela salva com sucesso em: ..\docs\metadata\n01_table_schema_report.csv


Não foram identificadas falhas nos schemas ou colunas para os arquivos da camada `interim`.

---

## 5. Validação de filtros de escopo

Nesta seção são verificados os anos e períodos de cada arquivo da camada `interim`, comparando com o período esperado definido pelas constantes **ANO_INICIO** e **ANO_FIM**.

In [ ]:
# Verificação de períodos aceitos pelo contrato de dados

full_table_path = TABLE_SCOPE_REPORT
relative_table_path = os.path.relpath(full_table_path)

queries_scope = {
    "continuidade_ano": f"SELECT COUNT(*) FROM read_parquet(?) WHERE AnoIndice NOT BETWEEN {ANO_INICIO} AND {ANO_FIM}",
    "continuidade_periodo": "SELECT COUNT(*) FROM read_parquet(?) WHERE NumPeriodoIndice NOT BETWEEN 1 AND 12",
    "continuidade_indicador": "SELECT COUNT(*) FROM read_parquet(?) WHERE SigIndicador NOT IN ('DEC', 'FEC') OR SigIndicador IS NULL",
    "interrupcoes_ano": f"SELECT COUNT(*) FROM read_parquet(?) WHERE AnoIndice NOT BETWEEN {ANO_INICIO} AND {ANO_FIM}",
    "limites_ano": f"SELECT COUNT(*) FROM read_parquet(?) WHERE AnoIndice NOT BETWEEN {ANO_INICIO} AND {ANO_FIM}",
    "limites_indicador": "SELECT COUNT(*) FROM read_parquet(?) WHERE SigIndicador NOT IN ('DEC', 'FEC') OR SigIndicador IS NULL",
}
fonts_scope = {
    "continuidade_ano": CONT_PATH,
    "continuidade_periodo": CONT_PATH,
    "continuidade_indicador": CONT_PATH,
    "interrupcoes_ano": INT_PATH,
    "limites_ano": LIM_PATH,
    "limites_indicador": LIM_PATH,
}
scope_results = []
for rule, query in queries_scope.items():
    failures = int(con.execute(query, [str(fonts_scope[rule])]).fetchone()[0])
    scope_results.append({"regra": rule, "falhas": failures})

df_scope = pd.DataFrame(scope_results)
display(df_scope)

df_scope.to_csv(
    full_table_path,
    index=False,
)

print(f"\nTabela salva com sucesso em: {relative_table_path}")

,regra,falhas
0,continuidade_ano,0
1,continuidade_periodo,0
2,continuidade_indicador,0
3,interrupcoes_ano,0
4,limites_ano,0
5,limites_indicador,0



Tabela salva com sucesso em: ..\docs\metadata\n01_table_scope_report.csv


Não foram identificadas falhas de granularidade ou de período nos arquivos interim, conforme as regras definidas no contrato de dados.

---

## 6. Validação de nulos e conversões

Nesta seção são validados os campos críticos nulos e conversões que resultaram em NULL.

In [ ]:
# Validações de campos críticos nulos e conversões que resultaram em NULL

full_table_path = TABLE_NULL_REPORT
relative_table_path = os.path.relpath(full_table_path)

null_queries = {
    "continuidade_ide_conjunto": (CONT_PATH, "IdeConjunto IS NULL"),
    "continuidade_valor": (CONT_PATH, "VlrIndiceEnviado IS NULL"),
    "interrupcoes_ide_conjunto": (INT_PATH, "IdeConjunto IS NULL"),
    "interrupcoes_inicio": (INT_PATH, "DatInicioInterrupcao IS NULL"),
    "interrupcoes_fim": (INT_PATH, "DatFimInterrupcao IS NULL"),
    "interrupcoes_duracao": (INT_PATH, "DuracaoHoras IS NULL"),
    "limites_ide_conjunto": (LIM_PATH, "IdeConjunto IS NULL"),
    "limites_valor": (LIM_PATH, "VlrLimite IS NULL"),
    "atributos_ide_conjunto": (ATR_PATH, "IdeConjunto IS NULL"),
    "regiao_ide_conjunto": (REG_PATH, "IdeConjunto IS NULL"),
}

null_results = []
for rule, (path, condition) in null_queries.items():
    total, null = con.execute(
        f"SELECT COUNT(*), COUNT(*) FILTER (WHERE {condition}) FROM read_parquet(?)",
        [str(path)],
    ).fetchone()
    percent = round((null / total * 100) if total else 0, 2)
    null_results.append(
        {
            "regra": rule,
            "total": total,
            "falhas": null,
            "percentual_falhas": percent,
        }
    )

df_null = pd.DataFrame(null_results)
display(df_null.style.format({"percentual_falhas": "{:.2f} %"}))

df_null.to_csv(
    full_table_path,
    index=False,
)

print(f"\nTabela salva com sucesso em: {relative_table_path}")

,regra,total,falhas,percentual_falhas
0,continuidade_ide_conjunto,375082,0,0.00 %
1,continuidade_valor,375082,0,0.00 %
2,interrupcoes_ide_conjunto,44410820,0,0.00 %
3,interrupcoes_inicio,44410820,0,0.00 %
4,interrupcoes_fim,44410820,0,0.00 %
5,interrupcoes_duracao,44410820,0,0.00 %
6,limites_ide_conjunto,31844,0,0.00 %
7,limites_valor,31844,0,0.00 %
8,atributos_ide_conjunto,7569,0,0.00 %
9,regiao_ide_conjunto,15162,0,0.00 %



Tabela salva com sucesso em: ..\docs\metadata\n01_table_null_report.csv


Nenhuma falha de integridade referencial foi detectada entre os datasets analisados.

---

## 7. Validação de granularidade e duplicidades

Nesta seção, são observadas as duplicidades presentes nas granularidades definidas pelo contrato de dados.

In [ ]:
# Duplicidades nas granularidades definidas pelo contrato

full_table_path = TABLE_DUPLICATES_REPORT
relative_table_path = os.path.relpath(full_table_path)

duplicate_queries = {
    "continuidade": (
        CONT_PATH,
        "IdeConjunto, AnoIndice, NumPeriodoIndice, SigIndicador",
    ),
    "limites": (LIM_PATH, "NumCNPJ, IdeConjunto, SigIndicador, AnoIndice"),
    "atributos": (ATR_PATH, "NumCNPJ, IdeConjunto"),
    "regiao": (REG_PATH, "IdeConjunto"),
}

duplicate_results = []
for name, (path, key) in duplicate_queries.items():
    query = f"""
        SELECT COUNT(*) FROM (
            SELECT {key}                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                
            FROM read_parquet(?)
            GROUP BY {key}
            HAVING COUNT(*) > 1
        )
    """
    fails = int(con.execute(query, [str(path)]).fetchone()[0])
    duplicate_results.append({"dataset": name, "falhas": fails})

df_duplicates = pd.DataFrame(duplicate_results)
display(df_duplicates)

df_duplicates.to_csv(
    full_table_path,
    index=False,
)

print(f"\nTabela salva com sucesso em: {relative_table_path}")

,dataset,falhas
0,continuidade,0
1,limites,0
2,atributos,0
3,regiao,0



Tabela salva com sucesso em: ..\docs\metadata\n01_table_duplicates_report.csv


Não foram identificados atributos com valores duplicados nas granularidades definidas pelo contrato de dados.

---

## 8. Validação de chaves e relacionamentos

Nesta seção, o código realiza uma verificação de duplicidades nos conjuntos de dados definidos pelo contrato. Ele define consultas SQL para cada conjunto de dados, agrupando por chaves específicas e contando quantos grupos têm mais de uma ocorrência. 

In [ ]:
# Diagnóstico complementar de relacionamentos

full_table_path = TABLE_RELATIONSHIP_REPORT
relative_table_path = os.path.relpath(full_table_path)

relationship_queries = {
    "continuidade_sem_regiao": f"""
        SELECT COUNT(*) FROM (
            SELECT DISTINCT c.IdeConjunto
            FROM read_parquet('{CONT_PATH}') c
            LEFT JOIN read_parquet('{REG_PATH}') r USING (IdeConjunto)
            WHERE r.IdeConjunto IS NULL
        )
    """,
    "interrupcoes_sem_regiao": f"""
        SELECT COUNT(*) FROM (
            SELECT DISTINCT i.IdeConjunto
            FROM read_parquet('{INT_PATH}') i
            LEFT JOIN read_parquet('{REG_PATH}') r USING (IdeConjunto)
            WHERE r.IdeConjunto IS NULL
        )
    """,
    "continuidade_sem_limite": f"""
        SELECT COUNT(*) FROM (
            SELECT DISTINCT c.IdeConjunto, c.SigIndicador, c.AnoIndice
            FROM read_parquet('{CONT_PATH}') c
            LEFT JOIN read_parquet('{LIM_PATH}') l
              ON c.IdeConjunto = l.IdeConjunto
             AND c.SigIndicador = l.SigIndicador
             AND c.AnoIndice = l.AnoIndice
            WHERE l.IdeConjunto IS NULL
        )
    """,
}

relationship_results = []
for rule, query in relationship_queries.items():
    fails = int(con.execute(query).fetchone()[0])
    relationship_results.append({"regra": rule, "falhas": fails})

df_relationships = pd.DataFrame(relationship_results)
display(df_relationships)

df_relationships.to_csv(
    full_table_path,
    index=False,
)

print(f"\nTabela salva com sucesso em: {relative_table_path}")

,regra,falhas
0,continuidade_sem_regiao,4
1,interrupcoes_sem_regiao,4
2,continuidade_sem_limite,64



Tabela salva com sucesso em: ..\docs\metadata\n01_table_relationship_report.csv


Foram identificadas chaves de relacionamento com falhas, indicando que alguns registros não possuem correspondência entre os datasets analisados.

---

## 9. Validações complementares por dataset

Neste tópico, é realizada uma série de verificações de qualidade de dados nos arquivos da camada `interim`, incluindo a validação de duplicidades, relacionamentos entre conjuntos de dados e conformidade com as regras contratuais definidas.

In [ ]:
# Validações complementares por dataset

full_table_path = TABLE_COMPLEMENTAR_VALIDATIONS
relative_table_path = os.path.relpath(full_table_path)

specific_queries = {
    "continuidade_valor_negativo": (CONT_PATH, "VlrIndiceEnviado < 0"),
    "interrupcoes_duracao_negativa": (INT_PATH, "DuracaoHoras < 0"),
    "interrupcoes_datas_invertidas": (
        INT_PATH,
        "DatFimInterrupcao < DatInicioInterrupcao",
    ),
    "interrupcoes_consumidores_negativos": (INT_PATH, "NumConsumidorConjunto < 0"),
    "regiao_uf_vazia": (REG_PATH, "UF IS NULL OR TRIM(UF) = ''"),
    "regiao_regiao_invalida": (
        REG_PATH,
        "Regiao NOT IN ('NORTE', 'NORDESTE', 'CENTRO-OESTE', 'SUDESTE', 'SUL', 'NAO INFORMADO')",
    ),
}

specific_results = []
for rule, (path, condition) in specific_queries.items():
    fails = int(
        con.execute(
            f"SELECT COUNT(*) FROM read_parquet(?) WHERE {condition}", [str(path)]
        ).fetchone()[0]
    )
    specific_results.append({"regra": rule, "falhas": fails})

df_specific_results = pd.DataFrame(specific_results)
display(df_specific_results)

df_specific_results.to_csv(
    full_table_path,
    index=False,
)

print(f"\nTabela salva com sucesso em: {relative_table_path}")

,regra,falhas
0,continuidade_valor_negativo,0
1,interrupcoes_duracao_negativa,0
2,interrupcoes_datas_invertidas,0
3,interrupcoes_consumidores_negativos,0
4,regiao_uf_vazia,29
5,regiao_regiao_invalida,0



Tabela salva com sucesso em: ..\docs\metadata\n01_table_complementar_validations.csv


Foram identificadas 29 regiões com UF vazia na base de dados.

---

## 10. Problemas encontrados e ações recomendadas

Neste tópico, são agrupadas as regras com falhas e é medido o impacto relativo de cada uma, considerando o total de linhas do dataset correspondente.


In [ ]:
# Detalha as regras com falhas e mede o impacto relativo

full_table_path = TABLE_ISSUES_REPORT
relative_table_path = os.path.relpath(full_table_path)

impact_queries = {
    "continuidade_sem_regiao": (
        f"SELECT COUNT(DISTINCT IdeConjunto) FROM read_parquet('{CONT_PATH}')",
        "A ausência regional é tratada como 'NÃO INFORMADO' no modelo"
        "e afeta apenas o enriquecimento geográfico.",
    ),
    "interrupcoes_sem_regiao": (
        f"SELECT COUNT(DISTINCT IdeConjunto) FROM read_parquet('{INT_PATH}')",
        "A ausência regional é tratada como 'NÃO INFORMADO' no modelo"
        "e afeta apenas o enriquecimento geográfico.",
    ),
    "continuidade_sem_limite": (
        f"SELECT COUNT(*) FROM (SELECT DISTINCT IdeConjunto, SigIndicador, AnoIndice FROM read_parquet('{CONT_PATH}'))",
        "A combinação sem limite não pode ser usada" "em comparação regulatória.",
    ),
    "regiao_uf_vazia": (
        f"SELECT COUNT(*) FROM read_parquet('{REG_PATH}')",
        "A UF vazia reduz o detalhamento geográfico;"
        "o modelo usa 'NÃO INFORMADO' quando necessário.",
    ),
    "continuidade_valor_negativo": (
        f"SELECT COUNT(*) FROM read_parquet('{CONT_PATH}')",
        "Valor negativo viola o domínio definido para o indicador.",
    ),
    "interrupcoes_duracao_negativa": (
        f"SELECT COUNT(*) FROM read_parquet('{INT_PATH}')",
        "Duração negativa indica inconsistência temporal.",
    ),
    "interrupcoes_datas_invertidas": (
        f"SELECT COUNT(*) FROM read_parquet('{INT_PATH}')",
        "Data final anterior ao início indica inconsistência temporal.",
    ),
    "interrupcoes_consumidores_negativos": (
        f"SELECT COUNT(*) FROM read_parquet('{INT_PATH}')",
        "Quantidade negativa de consumidores viola o domínio do campo.",
    ),
    "regiao_regiao_invalida": (
        f"SELECT COUNT(*) FROM read_parquet('{REG_PATH}')",
        "Região fora do domínio impede a classificação geográfica correta.",
    ),
}

issues = []
for table in [df_relationships, df_specific_results]:
    for _, row in table[table["falhas"] > 0].iterrows():
        rule = row["regra"]
        if rule not in impact_queries:
            continue
        total_query, justify = impact_queries[rule]
        total_references = int(con.execute(total_query).fetchone()[0])
        incidents = int(row["falhas"])
        issues.append(
            {
                "regra": rule,
                "ocorrencias": incidents,
                "total_referencia": total_references,
                "percentual_afetado": (
                    round(incidents / total_references * 100, 2)
                    if total_references
                    else 0
                ),
                "justificativa": justify,
            }
        )

if issues:
    df_issues = pd.DataFrame(issues)
    display(df_issues.style.format({"percentual_afetado": "{:.2f} %"}))
else:
    display(
        pd.DataFrame(
            {
                "result": [
                    "Nenhuma falha foi encontrada nas" "validações complementares."
                ]
            }
        )
    )

df_issues.to_csv(
    full_table_path,
    index=False,
)

print(f"\nTabela salva com sucesso em: {relative_table_path}")

,regra,ocorrencias,total_referencia,percentual_afetado,justificativa
0,continuidade_sem_regiao,4,3824,0.10 %,A ausência regional é tratada como 'NÃO INFORMADO' no modeloe afeta apenas o enriquecimento geográfico.
1,interrupcoes_sem_regiao,4,3766,0.11 %,A ausência regional é tratada como 'NÃO INFORMADO' no modeloe afeta apenas o enriquecimento geográfico.
2,continuidade_sem_limite,64,31292,0.20 %,A combinação sem limite não pode ser usadaem comparação regulatória.
3,regiao_uf_vazia,29,15162,0.19 %,A UF vazia reduz o detalhamento geográfico;o modelo usa 'NÃO INFORMADO' quando necessário.



Tabela salva com sucesso em: ..\docs\metadata\n01_table_issues_report.csv


As falhas identificadas não impedem o consumo da camada interim (valores inferiores a 1%), sendo documentadas apenas para fins de auditoria e melhoria contínua.